In [10]:
%run -n "/Users/alejandrogomez-paz/Desktop/RAG/rag_code/old_RAG.ipynb"

ModuleNotFoundError: No module named 'LLM_local'

ModuleNotFoundError: No module named 'LLM_local'

In [6]:
import json
import sys
sys.path.append(r"/Users/alejandrogomez-paz/Desktop/RAG/local_LLM")
sys.path.append(r"/Users/alejandrogomez-paz/Desktop/RAG/corpus_v1")

from rad_ai import query
from benchmark_golden_pairs_1 import golden_pairs, should_refuse_map, answer_keys
import time

'''
Quality:
  CoP - Context Precision: fraction of retrieved chunks that are golden
  CiP - Citation Precision: fraction of cited sources that are correct
  RR  - Refusal Rate: fraction of correct refusal decisions (maximize)
  HR  - Hallucination Rate: fraction of answers with unsupported claims (minimize)
Latency:
  RT  - mean retrieval time (s)
'''

CORPUS_PATH = r"/Users/alejandrogomez-paz/Desktop/RAG/corpus_v1/corpus_1.jsonl"
metas = [
    {"source": r["source"], "page": r["section"], "chunk_id": r["chunk_id"]}
    for r in (json.loads(l) for l in open(CORPUS_PATH, encoding="utf-8"))
    if r.get("record_type") == "chunk"
]

cid2sp   = {m["chunk_id"]: f'{m["source"]} | page {m["page"]}' for m in metas}
golden_sp = {q: {cid2sp[c] for c in ids} for q, ids in golden_pairs.items()}


def get_citation(hit) -> str:
    return f'{hit["source"]} | page {hit["page"]}'

def context_precision(query, retrieved_hits):
    if not retrieved_hits: return 0.0
    g = golden_sp.get(query, set())
    return sum(1 for h in retrieved_hits if get_citation(h) in g) / len(retrieved_hits)

def context_recall(query, retrieved_hits):        # Hit@k — the honest single-golden signal
    g = golden_sp.get(query, set())
    if not g: return None
    got = {get_citation(h) for h in retrieved_hits}
    return len(got & g) / len(g)

def citation_precision(cited_ids, golden_ids):    # cited_ids already (source,page) strings
    if not cited_ids: return 0.0
    return sum(1 for c in cited_ids if c in golden_ids) / len(cited_ids)

def refusal_score(did_refuse, should_refuse):
    correct = sum(1 for d, s in zip(did_refuse, should_refuse) if d == s)
    return correct / len(should_refuse)

def hallucination_rate(judgments):
    """judgments: list of bools, True = answer contained hallucination.
    v0: with an LLM judge."""
    if not judgments:
        return 0.0
    return sum(judgments) / len(judgments)

def quality_score(s):
    return (0.25 * s['COP']
          + 0.25 * s['CIP']
          + 0.25 * s['RR']          # already "higher = better"
          + 0.25 * (1 - s['HR']))

def judge_hallucination(q, answer, chunks):
    """LLM-as-judge. Returns True if the answer makes claims not supported by the retrieved chunks."""
    import json, re
    from rad_ai import query

    # a correct refusal cannot hallucinate
    if isinstance(answer, dict) and answer.get('refused'):
        return False

    # pull answer text
    if isinstance(answer, dict):
        ans = answer.get('answer') or answer.get('text') or answer.get('response') or json.dumps(answer)
    else:
        ans = str(answer)
    if not str(ans).strip():
        return False

    # build context from retrieved hits
    def chunk_text(h):
        for k in ('text', 'content', 'chunk', 'body', 'passage'):
            if isinstance(h, dict) and h.get(k):
                return str(h[k])
        return str(h)
    context = "\n\n".join(f"[{i+1}] {chunk_text(h)}" for i, h in enumerate(chunks)) or "(no context)"

    prompt = (
        "You are a strict factuality judge for a retrieval-augmented QA system. "
        "Given a QUESTION, the retrieved CONTEXT, and an ANSWER, decide whether the ANSWER "
        "contains a hallucination: any factual claim NOT supported by the CONTEXT. "
        "Judge support only against the CONTEXT, not outside knowledge. A refusal is NOT a hallucination.\n"
        'Respond with ONLY JSON: {"hallucinated": true|false}\n\n'
        f"QUESTION:\n{q}\n\nCONTEXT:\n{context}\n\nANSWER:\n{ans}\n"
    )

    try:
        raw = query(prompt)
        raw = raw if isinstance(raw, str) else str(raw)
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        if m:
            return bool(json.loads(m.group(0)).get('hallucinated', False))
        return bool(re.search(r'\btrue\b|\byes\b', raw, re.I))
    except Exception:
        return False  # judge failed -> don't penalize
    
REFUSAL = "don't have enough information"

def benchmark_v0(k=5):
    latencies, cop_list, crec_list, cip_list = [], [], [], []
    did_refuse, should_refuse, halluc = [], [], []

    for q, sr in should_refuse_map.items():
        t0 = time.time()
        hits = [{"source": m["source"], "page": m["page"],
                 "chunk_id": m["chunk_id"], "text": txt}
                for _, txt, m in retrieve(q, k=k)]
        latencies.append(time.time() - t0)

        answer = rag_answer(q, k=k)
        refused = REFUSAL in answer.lower()

        cop_list.append(context_precision(q, hits))
        r = context_recall(q, hits)
        if r is not None: crec_list.append(r)

        if not sr:  # CiP + hallucination judged on answerable queries only
            cited = [get_citation(h) for h in hits
                     if h["source"].lower() in answer.lower()]
            cip_list.append(citation_precision(cited, golden_sp.get(q, set())))
            halluc.append(False if refused else judge_hallucination(q, answer, hits))

        did_refuse.append(refused)
        should_refuse.append(sr)

    scores = {
        'COP': sum(cop_list)/len(cop_list),
        'REC': sum(crec_list)/len(crec_list),
        'CIP': sum(cip_list)/len(cip_list),
        'RR':  refusal_score(did_refuse, should_refuse),
        'HR':  hallucination_rate(halluc),
        'RT':  sum(latencies)/len(latencies),
    }
    return quality_score(scores), scores
    latencies, cop_list, cip_list = [], [], []
    did_refuse, should_refuse, halluc = [], [], []

    for sample in dataset:
        q = sample['query']

        t0 = time.time()
        chunks = rag.retrieve(q)
        latencies.append(time.time() - t0)

        answer = rag.generate(q, chunks)  # your bug: used undefined `retrieved_chunks`

        cop_list.append(context_precision(q, chunks, golden_pairs))
        cip_list.append(citation_precision(answer.get('citations', []),
                                           golden_pairs[q]))
        did_refuse.append(answer.get('refused', False))
        should_refuse.append(sample['should_refuse'])
        halluc.append(judge_hallucination(q, answer, chunks))  # you supply this

    scores = {
        'COP': sum(cop_list) / len(cop_list),
        'CIP': sum(cip_list) / len(cip_list),
        'RR':  refusal_score(did_refuse, should_refuse),
        'HR':  hallucination_rate(halluc),
        'RT':  sum(latencies) / len(latencies),
    }
    return quality_score(scores), scores

In [7]:
%run -n "/Users/alejandrogomez-paz/Desktop/RAG/rag_code/old_RAG.ipynb"

ModuleNotFoundError: No module named 'nbformat'